In [ ]:
# ===== 安装 PyTorch =====
!pip install torch==2.2.2 torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121

# ===== 安装 PyG依赖（版本必须匹配 2.2.2）=====
!pip install pyg-lib torch-scatter torch-sparse torch-cluster \
  -f https://data.pyg.org/whl/torch-2.2.2+cu121.html

# ===== 最后装 PyG =====
!pip install torch-geometric
!pip uninstall -y numpy
!pip install numpy==1.26.4

import torch
import torch_sparse
import pyg_lib
from torch_geometric.loader import NeighborLoader

print(torch.__version__)
print("全部OK")

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 116.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 72.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 147.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 19.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 44.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 19.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1

ModuleNotFoundError: No module named 'numpy.char'

In [ ]:
import numpy as np
import torch

print("numpy:", np.__version__)
print("torch:", torch.__version__)

a = np.array([1,2,3], dtype=np.float32)
b = torch.from_numpy(a)

print("OK:", b)

numpy: 1.26.4
torch: 2.2.2+cu121
OK: tensor([1., 2., 3.])


In [ ]:
import torch_sparse
import pyg_lib
from torch_geometric.loader import NeighborLoader

print("PyG OK")

PyG OK


In [ ]:
import numpy as np
import torch
import torch_sparse
import pyg_lib
from torch_geometric.loader import NeighborLoader

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

a = np.array([1,2,3], dtype=np.float32)
b = torch.from_numpy(a)

print("numpy->torch OK")
print("PyG OK")

numpy: 1.26.4
torch: 2.2.2+cu121
cuda: True
numpy->torch OK
PyG OK


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/CS-30/image_train_car.zip" -d /content/
!unzip -q "/content/drive/MyDrive/CS-30/image_train_scene.zip" -d /content/

In [ ]:
# ================================
# Vehicle ReID + Knowledge Graph
# Mini-batch Hetero GNN Training Script
# ================================

# 导入未来版本注解支持，方便类型提示写法更简洁
from __future__ import annotations

# 导入随机数相关库
import os
import random
import re

# 导入数据类装饰器
from dataclasses import dataclass

# 导入类型提示
from typing import Dict, List, Tuple

# 导入 PyTorch
import torch

# 导入 PyTorch 神经网络模块
import torch.nn as nn

# 导入 PyTorch 常用函数模块
import torch.nn.functional as F

# 导入 PIL 图片读取工具
from PIL import Image

# 导入 Dataset 和 DataLoader
from torch.utils.data import DataLoader, Dataset

# 导入 torchvision 的图像变换
from torchvision import transforms

# 导入 tqdm 进度条
from tqdm import tqdm

# 导入 PyG 的异构图数据结构
from torch_geometric.data import HeteroData

# 导入 PyG 的异构图卷积和 GATConv
from torch_geometric.nn import HeteroConv, GATConv

# 导入 PyG 的 NeighborLoader，用于子图训练
from torch_geometric.loader import NeighborLoader


# --------------------------------------------------
# 文件名解析正则
# 例如：0001_c001_00016450_0.jpg
# --------------------------------------------------
FILENAME_RE = re.compile(r"^(?P<vid>\d+)_c(?P<cid>\d+)_.*\.jpg$", re.IGNORECASE)


# --------------------------------------------------
# 固定随机种子，保证实验可复现
# --------------------------------------------------
def seed_everything(seed: int) -> None:
    # 固定 Python 随机数种子
    random.seed(seed)

    # 固定 hash 种子
    os.environ["PYTHONHASHSEED"] = str(seed)

    # 固定 CPU 上的 torch 随机种子
    torch.manual_seed(seed)

    # 固定所有 GPU 上的 torch 随机种子
    torch.cuda.manual_seed_all(seed)


# --------------------------------------------------
# 从文件名中解析车辆 ID 和摄像头 ID
# --------------------------------------------------
def parse_ids(filename: str) -> Tuple[int, str]:
    # 用正则匹配文件名
    m = FILENAME_RE.match(os.path.basename(filename))

    # 如果格式不对，直接报错
    if m is None:
        raise ValueError(f"Unexpected filename format: {filename}")

    # 解析车辆 ID，例如 0001 -> 1
    vehicle_id = int(m.group("vid"))

    # 解析摄像头 ID，例如 001 -> c001
    camera_id = f"c{m.group('cid')}"

    # 返回车辆 ID 和摄像头 ID
    return vehicle_id, camera_id


# --------------------------------------------------
# 配对数据集
# 每个样本包含：
# - car 图片路径
# - scene 图片路径
# - vehicle_id
# - camera_id
# - 文件名
# --------------------------------------------------
class ImagePairDataset(Dataset):
    # 初始化函数
    def __init__(self, car_dir: str, scene_dir: str):
        # 保存车辆图片目录
        self.car_dir = car_dir

        # 保存场景图片目录
        self.scene_dir = scene_dir

        # 读取 car_dir 下的所有图片文件名
        car_files = sorted(
            f for f in os.listdir(car_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
        )

        # 如果没有图片，直接报错
        if len(car_files) == 0:
            raise RuntimeError(f"No images found in: {car_dir}")

        # 检查 scene_dir 中是否存在所有同名文件
        missing = [f for f in car_files if not os.path.exists(os.path.join(scene_dir, f))]

        # 如果有缺失，报错
        if missing:
            raise RuntimeError(f"Scene folder missing {len(missing)} paired files. Example: {missing[0]}")

        # 保存文件名列表
        self.files = car_files

        # 预存每个文件对应的标签
        self.labels: List[int] = []

        # 预存每个文件对应的摄像头
        self.cameras: List[str] = []

        # 遍历所有文件
        for name in self.files:
            # 解析车辆 ID 和摄像头 ID
            vid, cid = parse_ids(name)

            # 保存车辆 ID
            self.labels.append(vid)

            # 保存摄像头 ID
            self.cameras.append(cid)

    # 返回数据集长度
    def __len__(self) -> int:
        return len(self.files)

    # 返回单个样本
    def __getitem__(self, idx: int):
        # 取文件名
        name = self.files[idx]

        # 拼接 car 图片路径
        car_path = os.path.join(self.car_dir, name)

        # 拼接 scene 图片路径
        scene_path = os.path.join(self.scene_dir, name)

        # 取车辆 ID
        vehicle_id = self.labels[idx]

        # 取摄像头 ID
        camera_id = self.cameras[idx]

        # 返回完整样本
        return car_path, scene_path, vehicle_id, camera_id, name


# --------------------------------------------------
# 构建 DINOv2 所需的图像变换
# --------------------------------------------------
def build_transforms(image_size: int) -> transforms.Compose:
    # 返回组合变换
    return transforms.Compose(
        [
            # resize 到固定大小
            transforms.Resize((image_size, image_size)),

            # 转为 tensor
            transforms.ToTensor(),

            # ImageNet 标准归一化
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )


# --------------------------------------------------
# 读取单张图片
# --------------------------------------------------
def load_image(path: str) -> Image.Image:
    # 以 RGB 模式打开图片
    img = Image.open(path).convert("RGB")

    # 返回图片对象
    return img


# --------------------------------------------------
# 简单 MLP：768 -> 512 -> 256
# --------------------------------------------------
class MLP(nn.Module):
    # 初始化函数
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        # 调用父类初始化
        super().__init__()

        # 定义网络结构
        self.net = nn.Sequential(
            # 第一层线性映射
            nn.Linear(in_dim, hidden_dim),

            # ReLU 激活
            nn.ReLU(inplace=True),

            # 第二层线性映射
            nn.Linear(hidden_dim, out_dim),
        )

    # 前向传播
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 返回网络输出
        return self.net(x)


# --------------------------------------------------
# 异构图 GAT
# 这里保持和你原来风格尽量一致，只补上反向边
# --------------------------------------------------
class HeteroGAT(nn.Module):
    # 初始化函数
    def __init__(self, hidden_dim: int, heads: int, dropout: float):
        # 调用父类初始化
        super().__init__()

        # 第一层每种关系对应一个 GATConv
        conv1 = {
            # image -> camera
            ("image", "to", "camera"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),

            # image -> scene
            ("image", "to", "scene"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),

            # camera -> image
            ("camera", "to", "image"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),

            # scene -> image
            ("scene", "to", "image"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),
        }

        # 第一层异构卷积
        self.conv1 = HeteroConv(conv1, aggr="sum")

        # 第二层每种关系对应一个 GATConv
        conv2 = {
            # image -> camera
            ("image", "to", "camera"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),

            # image -> scene
            ("image", "to", "scene"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),

            # camera -> image
            ("camera", "to", "image"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),

            # scene -> image
            ("scene", "to", "image"): GATConv(
                (-1, -1),
                hidden_dim,
                heads=heads,
                concat=False,
                dropout=dropout,
                add_self_loops=False,
            ),
        }

        # 第二层异构卷积
        self.conv2 = HeteroConv(conv2, aggr="sum")

    # 前向传播
    def forward(self, x_dict, edge_index_dict):
        # 第一层卷积
        x_dict = self.conv1(x_dict, edge_index_dict)

        # 每类节点都做 ReLU
        x_dict = {k: F.relu(v) for k, v in x_dict.items()}

        # 第二层卷积
        x_dict = self.conv2(x_dict, edge_index_dict)

        # 返回所有节点类型的新表示
        return x_dict


# --------------------------------------------------
# 各种映射的打包类
# --------------------------------------------------
@dataclass
class GraphMappings:
    # image 文件名 -> image 节点索引
    image_id_map: Dict[str, int]

    # scene 文件名 -> scene 节点索引
    scene_id_map: Dict[str, int]

    # camera 名称 -> camera 节点索引
    camera_id_map: Dict[str, int]

    # image 文件名 -> 车辆 ID
    image_label_map: Dict[str, int]

    # image 文件名 -> 摄像头 ID
    image_camera_map: Dict[str, str]


# --------------------------------------------------
# 构建各种映射
# --------------------------------------------------
def build_mappings(dataset: ImagePairDataset) -> GraphMappings:
    # 初始化 image 节点映射
    image_id_map: Dict[str, int] = {}

    # 初始化 scene 节点映射
    scene_id_map: Dict[str, int] = {}

    # 初始化 camera 节点映射
    camera_id_map: Dict[str, int] = {}

    # 初始化 image -> vehicle_id 映射
    image_label_map: Dict[str, int] = {}

    # 初始化 image -> camera_id 映射
    image_camera_map: Dict[str, str] = {}

    # 遍历所有文件
    for idx, name in enumerate(dataset.files):
        # 当前 image 节点索引
        image_id_map[name] = idx

        # 当前 scene 节点索引（仍保持一一对应）
        scene_id_map[name] = idx

        # 解析车辆 ID 和摄像头 ID
        vehicle_id, camera_id = parse_ids(name)

        # 保存标签
        image_label_map[name] = vehicle_id

        # 保存 image -> camera
        image_camera_map[name] = camera_id

        # 如果是新摄像头，分配新索引
        if camera_id not in camera_id_map:
            camera_id_map[camera_id] = len(camera_id_map)

    # 返回封装后的映射对象
    return GraphMappings(
        image_id_map=image_id_map,
        scene_id_map=scene_id_map,
        camera_id_map=camera_id_map,
        image_label_map=image_label_map,
        image_camera_map=image_camera_map,
    )


# --------------------------------------------------
# 用 DataLoader 批量预计算 DINO 768 维特征
# mode = "car" 或 "scene"
# 返回 [N, 768] 的 CPU Tensor
# --------------------------------------------------
def compute_dino_features_once(
    dataset: ImagePairDataset,
    dinov2: nn.Module,
    tfm: transforms.Compose,
    device: torch.device,
    batch_size: int,
    num_workers: int,
    mode: str,
) -> torch.Tensor:
    # 构建 DataLoader
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    # 用于保存全部特征
    all_feats: List[torch.Tensor] = []

    # 设为 eval
    dinov2.eval()

    # 不计算梯度
    with torch.no_grad():
        # 遍历 batch
        for car_path, scene_path, vehicle_id, camera_id, name in tqdm(loader, desc=f"Precompute DINO {mode}"):
            # 临时保存本 batch 图片
            imgs = []

            # 如果是 car 模式
            if mode == "car":
                # 遍历 car 路径
                for p in car_path:
                    # 读图 + 预处理
                    imgs.append(tfm(load_image(p)))
            else:
                # 否则就是 scene 模式
                for p in scene_path:
                    # 读图 + 预处理
                    imgs.append(tfm(load_image(p)))

            # 堆叠为 batch tensor
            x = torch.stack(imgs, dim=0).to(device, non_blocking=True)

            # 用 DINOv2 提 768 特征
            feats768 = dinov2(x)

            # 保存到 CPU
            all_feats.append(feats768.detach().cpu())

    # 拼接成 [N, 768]
    return torch.cat(all_feats, dim=0)


# --------------------------------------------------
# 根据固定的 scene 768 特征，计算固定的 scene 256 特征
# 返回 [N, 256] CPU Tensor
# --------------------------------------------------
def compute_scene_features_once_from_base(
    scene_base_x_cpu: torch.Tensor,
    scene_mlp: nn.Module,
    device: torch.device,
    batch_size: int,
) -> torch.Tensor:
    # 设为 eval
    scene_mlp.eval()

    # 保存结果
    out_list: List[torch.Tensor] = []

    # 不计算梯度
    with torch.no_grad():
        # 按 batch 切分
        for start in tqdm(range(0, scene_base_x_cpu.size(0), batch_size), desc="Precompute fixed scene 256"):
            # 终点索引
            end = min(start + batch_size, scene_base_x_cpu.size(0))

            # 取这一段 base 特征
            x = scene_base_x_cpu[start:end].to(device)

            # 过 scene_mlp
            y = scene_mlp(x)

            # 保存到 CPU
            out_list.append(y.detach().cpu())

    # 拼接成 [N, 256]
    return torch.cat(out_list, dim=0)


# --------------------------------------------------
# 构建标签 tensor
# 把 vehicle_id 映射成连续类别 0..K-1
# --------------------------------------------------
def build_label_tensor(mappings: GraphMappings) -> Tuple[torch.Tensor, Dict[int, int]]:
    # 按 image 索引排序
    names_sorted_by_idx = sorted(mappings.image_id_map.items(), key=lambda kv: kv[1])

    # 取原始 vehicle id 列表
    vids = [mappings.image_label_map[name] for name, _ in names_sorted_by_idx]

    # 取去重后的 vehicle id
    uniq = sorted(set(vids))

    # 构建原始 vehicle_id -> 连续类别索引
    vid_to_class = {vid: i for i, vid in enumerate(uniq)}

    # 构建标签 tensor
    y = torch.tensor([vid_to_class[v] for v in vids], dtype=torch.long)

    # 返回标签 tensor 和映射
    return y, vid_to_class


# --------------------------------------------------
# 从完整图中采样的 image embedding 里构造 triplet
# --------------------------------------------------
def sample_batch_hard_triplets(
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    num_triplets: int,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # 获取设备
    device = embeddings.device

    # 把 labels 放到同一设备
    labels = labels.to(device)

    # 当前 batch 节点数
    N = embeddings.size(0)

    # 类别 -> 索引列表
    class_to_indices: Dict[int, List[int]] = {}

    # 遍历所有样本
    for i in range(N):
        # 当前类别
        c = int(labels[i].item())

        # 加入对应列表
        class_to_indices.setdefault(c, []).append(i)

    # 所有类别
    classes = list(class_to_indices.keys())

    # 如果类别数小于 2，无法构造 triplet
    if len(classes) < 2:
        raise RuntimeError("Triplet loss requires at least 2 classes in the sampled batch.")

    # 保存 anchor 索引
    anchors = []

    # 保存 positive 索引
    positives = []

    # 保存 negative 索引
    negatives = []

    # 循环采样
    for _ in range(num_triplets):
        # 随机挑一个 anchor
        a_idx = random.randrange(N)

        # anchor 类别
        a_cls = int(labels[a_idx].item())

        # positive 候选池
        pos_pool = class_to_indices[a_cls]

        # 如果该类在当前 batch 里只有 1 个样本，跳过
        if len(pos_pool) < 2:
            continue

        # 初始化 positive
        p_idx = a_idx

        # 确保 positive 不是 anchor 本身
        while p_idx == a_idx:
            p_idx = random.choice(pos_pool)

        # 初始化 negative 类别
        neg_cls = a_cls

        # 采样一个不同类别
        while neg_cls == a_cls:
            neg_cls = random.choice(classes)

        # 从负类里随机选一个
        n_idx = random.choice(class_to_indices[neg_cls])

        # 保存
        anchors.append(a_idx)
        positives.append(p_idx)
        negatives.append(n_idx)

    # 如果一个都没采到，返回空 tensor
    if len(anchors) == 0:
        empty = torch.empty((0, embeddings.size(1)), device=device)
        return empty, empty, empty

    # 构建 anchor embedding
    a = embeddings[torch.tensor(anchors, device=device)]

    # 构建 positive embedding
    p = embeddings[torch.tensor(positives, device=device)]

    # 构建 negative embedding
    n = embeddings[torch.tensor(negatives, device=device)]

    # 返回 triplet
    return a, p, n


# --------------------------------------------------
# 构建完整异构图
# 这里不再把 image.x 设置成训练输入
# 而是存 base_x / fixed_x / node_id / y
# 训练时在子图 batch 里动态生成真正的 x
# --------------------------------------------------
def build_full_heterodata(
    image_base_x_cpu: torch.Tensor,
    scene_fixed_x_cpu: torch.Tensor,
    y_cpu: torch.Tensor,
    mappings: GraphMappings,
) -> HeteroData:
    # 创建异构图对象
    data = HeteroData()

    # image 节点数量
    num_images = image_base_x_cpu.size(0)

    # scene 节点数量
    num_scenes = scene_fixed_x_cpu.size(0)

    # camera 节点数量
    num_cameras = len(mappings.camera_id_map)

    # 保存 image 的原始 768 维 base 特征
    data["image"].base_x = image_base_x_cpu

    # 先放一个占位 x，NeighborLoader 更稳妥
    data["image"].x = torch.zeros(num_images, 256, dtype=torch.float)

    # 保存 image 标签
    data["image"].y = y_cpu

    # 保存 image 节点原始索引
    data["image"].node_id = torch.arange(num_images, dtype=torch.long)

    # 保存 scene 固定 256 维特征
    data["scene"].fixed_x = scene_fixed_x_cpu

    # scene 占位 x
    data["scene"].x = scene_fixed_x_cpu.clone()

    # 保存 scene 原始索引
    data["scene"].node_id = torch.arange(num_scenes, dtype=torch.long)

    # camera 节点只保存 node_id
    data["camera"].node_id = torch.arange(num_cameras, dtype=torch.long)

    # camera 占位 x
    data["camera"].x = torch.zeros(num_cameras, 256, dtype=torch.float)

    # 构建 image -> camera 边
    src_img = []

    # 构建 image -> camera 边的目标端
    dst_cam = []

    # 遍历每张 image
    for name, img_idx in mappings.image_id_map.items():
        # 取对应 camera 名称
        cam_key = mappings.image_camera_map[name]

        # 取对应 camera 索引
        cam_idx = mappings.camera_id_map[cam_key]

        # 保存边起点
        src_img.append(img_idx)

        # 保存边终点
        dst_cam.append(cam_idx)

    # 构建 image -> camera edge_index
    edge_index_img_cam = torch.tensor([src_img, dst_cam], dtype=torch.long)

    # 存入图中
    data["image", "to", "camera"].edge_index = edge_index_img_cam

    # 构建 image -> scene 边
    src_img2 = list(range(num_images))

    # 因为是一一对应，所以 scene 也是同索引
    dst_scene = list(range(num_images))

    # 构建 edge_index
    edge_index_img_scene = torch.tensor([src_img2, dst_scene], dtype=torch.long)

    # 存入图中
    data["image", "to", "scene"].edge_index = edge_index_img_scene

    # 构建 camera -> image 反向边
    edge_index_cam_img = torch.stack([edge_index_img_cam[1], edge_index_img_cam[0]], dim=0)

    # 存入图中
    data["camera", "to", "image"].edge_index = edge_index_cam_img

    # 构建 scene -> image 反向边
    edge_index_scene_img = torch.stack([edge_index_img_scene[1], edge_index_img_scene[0]], dim=0)

    # 存入图中
    data["scene", "to", "image"].edge_index = edge_index_scene_img

    # 返回完整图
    return data


# --------------------------------------------------
# 给一个采样出来的子图 batch 填充真正训练要用的 x
# image.x = image_mlp(image.base_x)
# scene.x = scene.fixed_x
# camera.x = camera_emb(camera.node_id)
# --------------------------------------------------
def materialize_batch_features(
    batch: HeteroData,
    image_mlp: nn.Module,
    camera_emb: nn.Embedding,
    device: torch.device,
) -> HeteroData:
    # 把 batch 放到 device
    batch = batch.to(device)

    # image.base_x -> image_mlp -> image.x
    batch["image"].x = image_mlp(batch["image"].base_x)

    # scene 使用固定特征
    batch["scene"].x = batch["scene"].fixed_x

    # camera 使用可训练 embedding
    batch["camera"].x = camera_emb(batch["camera"].node_id)

    # 返回处理后的 batch
    return batch


# --------------------------------------------------
# 主函数
# --------------------------------------------------
def main():
    # 固定参数，适合 Colab 直接运行
    class Args:
        # 车辆图片目录
        car_dir = "/content/image_train_car"

        # 场景图片目录
        scene_dir = "/content/image_train_scene"

        # 训练轮数
        epochs = 1

        # 学习率
        lr = 1e-4

        # triplet loss 权重
        lambda_triplet = 0.5

        # 输入图片大小
        image_size = 224

        # DINO 预计算 batch size
        batch_size_feat = 8

        # DataLoader 线程数
        num_workers = 0

        # GAT 头数
        gat_heads = 1

        # GAT dropout
        gat_dropout = 0.1

        # 每个 mini-batch 采样多少 triplet
        triplets_per_epoch = 256

        # 随机种子
        seed = 42

        # checkpoint 保存路径
        save_path = "/content/checkpoint.pth"

        # NeighborLoader 的输入 image batch 大小
        train_batch_size = 512

        # 每层采样邻居数
        num_neighbors = [10, 10]

    # 实例化参数对象
    args = Args()

    # 固定随机种子
    seed_everything(args.seed)

    # 清理缓存，减少显存碎片
    torch.cuda.empty_cache()

    # 自动选择设备
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 打印设备信息
    print(f"[Info] device = {device}")

    # 构建配对数据集
    dataset = ImagePairDataset(args.car_dir, args.scene_dir)

    # 构建映射
    mappings = build_mappings(dataset)

    # 构建标签 tensor
    y_cpu, vid_to_class = build_label_tensor(mappings)

    # 类别数
    num_classes = len(vid_to_class)

    # 图片数
    num_images = len(dataset)

    # 摄像头数
    num_cameras = len(mappings.camera_id_map)

    # 打印数据规模
    print(f"[Info] images={num_images}, cameras={num_cameras}, classes={num_classes}")

    # 构建图像预处理
    tfm = build_transforms(args.image_size)

    # 提示开始加载 DINO
    print("[Info] Loading DINOv2 from torch.hub...")

    # 从 torch.hub 加载 DINOv2
    dinov2 = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")

    # 放到设备上
    dinov2 = dinov2.to(device)

    # 设为 eval
    dinov2.eval()

    # 冻结 DINOv2 参数
    for p in dinov2.parameters():
        p.requires_grad = False

    # 定义 image 分支 MLP（可训练）
    image_mlp = MLP(768, 512, 256).to(device)

    # 定义 scene 分支 MLP（冻结）
    scene_mlp = MLP(768, 512, 256).to(device)

    # 冻结 scene MLP 参数
    for p in scene_mlp.parameters():
        p.requires_grad = False

    # scene MLP 设为 eval
    scene_mlp.eval()

    # 预计算所有 car 图的 DINO 768 特征
    print("[Info] Precomputing all image base features (768)...")
    image_base_x_cpu = compute_dino_features_once(
        dataset=dataset,
        dinov2=dinov2,
        tfm=tfm,
        device=device,
        batch_size=args.batch_size_feat,
        num_workers=args.num_workers,
        mode="car",
    )

    # 预计算所有 scene 图的 DINO 768 特征
    print("[Info] Precomputing all scene base features (768)...")
    scene_base_x_cpu = compute_dino_features_once(
        dataset=dataset,
        dinov2=dinov2,
        tfm=tfm,
        device=device,
        batch_size=args.batch_size_feat,
        num_workers=args.num_workers,
        mode="scene",
    )

    # 用冻结的 scene_mlp 把 scene 768 变成固定 256
    print("[Info] Precomputing fixed scene node features (256)...")
    scene_fixed_x_cpu = compute_scene_features_once_from_base(
        scene_base_x_cpu=scene_base_x_cpu,
        scene_mlp=scene_mlp,
        device=device,
        batch_size=args.batch_size_feat * 8,
    )

    # 构建完整异构图
    print("[Info] Building full hetero graph...")
    data = build_full_heterodata(
        image_base_x_cpu=image_base_x_cpu,
        scene_fixed_x_cpu=scene_fixed_x_cpu,
        y_cpu=y_cpu,
        mappings=mappings,
    )

    # 构建 camera embedding（可训练）
    camera_emb = nn.Embedding(num_cameras, 256).to(device)

    # 构建异构 GAT
    gnn = HeteroGAT(
        hidden_dim=256,
        heads=args.gat_heads,
        dropout=args.gat_dropout,
    ).to(device)

    # 构建分类头
    classifier = nn.Linear(256, num_classes).to(device)

    # 定义交叉熵损失
    ce_loss = nn.CrossEntropyLoss()

    # 定义 triplet loss
    triplet_loss_fn = nn.TripletMarginLoss(margin=0.3, p=2)

    # 收集可训练参数
    optim_params = []

    # image_mlp 参数可训练
    optim_params += list(image_mlp.parameters())

    # camera embedding 参数可训练
    optim_params += list(camera_emb.parameters())

    # GNN 参数可训练
    optim_params += list(gnn.parameters())

    # 分类头参数可训练
    optim_params += list(classifier.parameters())

    # 构建优化器
    optimizer = torch.optim.Adam(optim_params, lr=args.lr)

    # 指定输入节点：只从 image 节点采样训练 batch
    input_nodes = ("image", torch.arange(data["image"].num_nodes))

    # 构建 NeighborLoader，实现子图训练
    train_loader = NeighborLoader(
        data=data,
        num_neighbors=args.num_neighbors,
        input_nodes=input_nodes,
        batch_size=args.train_batch_size,
        shuffle=True,
    )

    # 开始训练
    for epoch in range(1, args.epochs + 1):
        # GNN 设为训练模式
        gnn.train()

        # image_mlp 设为训练模式
        image_mlp.train()

        # classifier 设为训练模式
        classifier.train()

        # 记录本轮总损失
        total_loss = 0.0

        # 记录本轮总分类损失
        total_cls = 0.0

        # 记录本轮总 triplet 损失
        total_tri = 0.0

        # 记录 batch 数
        total_batches = 0

        # 按 mini-batch 子图训练
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{args.epochs}"):
            # 梯度清零
            optimizer.zero_grad(set_to_none=True)

            # 给当前子图填充真正的 x
            batch = materialize_batch_features(
                batch=batch,
                image_mlp=image_mlp,
                camera_emb=camera_emb,
                device=device,
            )

            # 前向传播
            out_dict = gnn(batch.x_dict, batch.edge_index_dict)

            # 取 image 节点输出 embedding
            img_emb_all = out_dict["image"]

            # NeighborLoader 返回的前 batch_size 个 image 是目标节点
            seed_count = batch["image"].batch_size

            # 只对目标 image 节点做监督
            img_emb = img_emb_all[:seed_count]

            # 取目标节点标签
            y = batch["image"].y[:seed_count].to(device)

            # 分类头输出 logits
            logits = classifier(img_emb)

            # 计算分类损失
            cls = ce_loss(logits, y)

            # 采样 triplet
            try:
                a, p, n = sample_batch_hard_triplets(
                    embeddings=img_emb,
                    labels=y,
                    num_triplets=args.triplets_per_epoch,
                )
            except RuntimeError:
                # 如果当前 batch 类别数不够，triplet 直接置 0
                a = torch.empty((0, img_emb.size(1)), device=device)
                p = torch.empty((0, img_emb.size(1)), device=device)
                n = torch.empty((0, img_emb.size(1)), device=device)

            # 如果 triplet 为空
            if a.numel() == 0:
                # triplet loss 设为 0
                tri = torch.tensor(0.0, device=device)
            else:
                # 否则正常计算 triplet loss
                tri = triplet_loss_fn(a, p, n)

            # 总损失
            loss = cls + args.lambda_triplet * tri

            # 反向传播
            loss.backward()

            # 更新参数
            optimizer.step()

            # 累加损失
            total_loss += loss.item()

            # 累加分类损失
            total_cls += cls.item()

            # 累加 triplet 损失
            total_tri += tri.item()

            # batch 数加一
            total_batches += 1

        # 计算平均总损失
        avg_loss = total_loss / max(total_batches, 1)

        # 计算平均分类损失
        avg_cls = total_cls / max(total_batches, 1)

        # 计算平均 triplet 损失
        avg_tri = total_tri / max(total_batches, 1)

        # 打印 epoch 日志
        print(
            f"[Epoch {epoch:03d}/{args.epochs}] "
            f"loss={avg_loss:.4f} "
            f"cls={avg_cls:.4f} "
            f"tri={avg_tri:.4f}"
        )

    # 构建 checkpoint
    ckpt = {
        # 保存 image_mlp 权重
        "image_mlp": image_mlp.state_dict(),

        # 保存 scene_mlp 权重（虽然冻结，也保存）
        "scene_mlp": scene_mlp.state_dict(),

        # 保存 camera embedding 权重
        "camera_embedding": camera_emb.state_dict(),

        # 保存 hetero gnn 权重
        "hetero_gnn": gnn.state_dict(),

        # 保存分类头权重
        "classifier": classifier.state_dict(),

        # 保存优化器状态
        "optimizer": optimizer.state_dict(),

        # 保存元信息
        "meta": {
            "num_classes": num_classes,
            "vid_to_class": vid_to_class,
            "camera_id_map": mappings.camera_id_map,
            "image_size": args.image_size,
            "dinov2_name": "dinov2_vitb14",
            "num_neighbors": args.num_neighbors,
            "train_batch_size": args.train_batch_size,
        },
    }

    # 保存 checkpoint 到指定路径
    torch.save(ckpt, args.save_path)

    # 打印保存信息
    print(f"[Info] Saved checkpoint to: {args.save_path}")


# --------------------------------------------------
# 程序入口
# --------------------------------------------------
if __name__ == "__main__":
    # 调用主函数
    main()

[Info] device = cuda
[Info] images=37778, cameras=20, classes=576
[Info] Loading DINOv2 from torch.hub...


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth
100%|██████████| 330M/330M [00:01<00:00, 299MB/s]


[Info] Precomputing all image base features (768)...


Precompute DINO car: 100%|██████████| 4723/4723 [03:32<00:00, 22.24it/s]


[Info] Precomputing all scene base features (768)...


...

[Info] Precomputing fixed scene node features (256)...


Precompute fixed scene 256: 100%|██████████| 591/591 [00:00<00:00, 3677.16it/s]

[Info] Building full hetero graph...



Epoch 1/1: 100%|██████████| 74/74 [00:02<00:00, 27.03it/s]


[Epoch 001/1] loss=6.4275 cls=6.2807 tri=0.2937
[Info] Saved checkpoint to: /content/checkpoint.pth
